# 07 — Body image preprocessing & quality

**Research question:** Có thể phát hiện lỗi kỹ thuật của silhouette và chuẩn hóa hai view mà giữ tỷ lệ hình thể không?

Dataset BodyM; baseline binary threshold → bbox → aspect-preserving letterbox 128×96. Không thực hiện RGB segmentation hay chứng nhận pose.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Module_01_User_and_Body_Profile.md").exists())
for path in [ROOT, ROOT / "scripts"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from IPython.display import display, Markdown
import numpy as np
import matplotlib.pyplot as plt


In [2]:
from PIL import Image
from bodym_cnn import preprocess_silhouette
from src.body_vision.quality import assess_mask
from run_bodym_baseline import read_csv
config = json.loads((ROOT / "configs/bodym_phase2.json").read_text())
cnn = json.loads((ROOT / "configs/bodym_cnn.json").read_text())
row = sorted(read_csv(ROOT / "data/raw/bodym/train/subject_to_photo_map.csv"), key=lambda r: (r["subject_id"], r["photo_id"]))[0]
fig, axes = plt.subplots(2, 2, figsize=(7, 9))
for i, view in enumerate(("mask", "mask_left")):
    path = ROOT / "data/raw/bodym/train" / view / (row["photo_id"] + ".png")
    with Image.open(path) as image:
        pixels = np.asarray(image.convert("L"))
    quality = assess_mask(pixels, config["quality"])
    print(view, quality)
    normalized = preprocess_silhouette(path, cnn)[0]
    assert normalized.shape == (128, 96) and set(np.unique(normalized)) <= {0, 1}
    axes[i, 0].imshow(pixels, cmap="gray"); axes[i, 0].set_title(view + " raw")
    axes[i, 1].imshow(normalized, cmap="gray"); axes[i, 1].set_title("128x96 letterbox")
for ax in axes.ravel(): ax.axis("off")
fig.tight_layout(); plt.show()

mask {'width': 720, 'height': 960, 'foreground_pixels': 120632, 'foreground_fraction': 0.17452546296296295, 'nonbinary_fraction': 0.0, 'bbox': [102, 63, 603, 889], 'largest_component_fraction': 1.0, 'components': 1, 'flags': [], 'status': 'passed_mask_checks', 'usable': True}
mask_left {'width': 720, 'height': 960, 'foreground_pixels': 77594, 'foreground_fraction': 0.11225983796296296, 'nonbinary_fraction': 0.0, 'bbox': [289, 60, 436, 906], 'largest_component_fraction': 1.0, 'components': 1, 'flags': [], 'status': 'passed_mask_checks', 'usable': True}


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6176\449722164.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


In [3]:
manifest = json.loads((ROOT / "data/manifests/bodym_phase2_v1.json").read_text())
rows = ["| Split/view | Images | Requires review | Unusable |", "|---|---:|---:|---:|"]
for name, q in manifest["quality_summary"].items():
    rows.append(f"| {name} | {q['images']} | {q['review_images']} | {q['unusable_images']} |")
display(Markdown("\n".join(rows)))

| Split/view | Images | Requires review | Unusable |
|---|---:|---:|---:|
| train/mask | 6134 | 123 | 0 |
| train/mask_left | 6134 | 1691 | 0 |
| testA/mask | 1684 | 34 | 0 |
| testA/mask_left | 1684 | 596 | 0 |
| testB/mask | 1160 | 55 | 0 |
| testB/mask_left | 1160 | 545 | 0 |

## Experiment / metrics / error analysis

Quy tắc cố định trước đánh giá: mask rỗng/full, kích thước foreground thấp, chạm khung, fragmentation, pixel không nhị phân. Cờ không tự loại ảnh khỏi metrics. Ảnh chạm khung không nhất thiết thiếu cơ thể; một silhouette sai pose vẫn có thể vượt qua mọi kiểm tra kỹ thuật.

Stress tests áp dụng trên mask gốc rồi đi qua chính preprocessing của checkpoint. Translation có padding phải cho cùng tensor; co/giãn biên, xoay và mất phần trên có thể làm thay đổi dự đoán. Đây không phải kiểm chứng pose/ánh sáng RGB hoặc camera thực tế.